# Mathematical Model of Innate Immune Response to Inhaled Toxicants
**Supplementary Material**

Authors: Aamy Bakry, Emma Brashear, Jacob Brasher, Shannon Z. Jones, Marcella Torres

Corresponding Author: Marcella Torres, mtorres@richmond.edu

Paper Reference: Manuscript submitted to Journal of Theoretical Biology (2024)

**Overview**

This notebook contains the computational implementation of our mathematical model describing the innate immune response to inhaled toxicants. The model captures interactions between epithelial cells, macrophages, and cytokines, with a focus on the transition from Type I to Type II inflammatory responses.

**Requirements**



```
import tellurium as te  # version 2.2.10
import numpy as np      # version 1.24.4
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)
```

**Contents**

1.   Epithelial-Cytokine Submodel (Produces Figures 4-5)
2.   M1-M2-Cytokine Submodel (Produces Figures 6-8)
3.   Full Combined Model (Produces Figure 9)
4.   Parameter sensitivity analysis for each model
5.   Model validation using experimental data for subsystems

**Usage Instructions**

1.   Install required packages
2.   Run cells individually or in order
3.   Figures will be saved in the working directory






Required Packages

In [ ]:
!pip install tellurium
!pip install numpy==1.24.4
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Epithelial-Cytokine Submodel

This section implements the epithelial cell and cytokine dynamics described in Equations 1-3 of the main text. The model tracks:


*   Healthy epithelial cells (E_h)
*   Damaged epithelial cells (E_d)
*   Pro-inflammatory cytokines (P)



Implementation

In [ ]:
#Epithelial-cytokine model
# Wood smoke exposure model with continuous exposure to simulate epithelial and cytokine dynamics for steady state analysis and validation to data
# Produces Figure 4
import tellurium as te
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Model and simulation code
continuous_exposure_model = '''
model wood_smoke_continuous_exposure
  // Species
  species E_h, E_d, P

   // Parameters
  E_h = 0.25;  // healthy epithelial cells
  E_d = 0.0;  // damaged epithelial cells
  P = 0.0;    // pro-inflammatory cytokines
  e1 = 0.0315;  // natural proliferation rate constant
  e2 = 0.001;   // natural death rate constant
  e3 = 0.005;   // damage rate constant due to wood smoke
  e4 = 0.002;   // repair rate constant
  e5 = 0.1;     // cytokine production rate constant by healthy cells
  e6 = 0.2;     // cytokine production rate constant by damaged cells
  e7 = 0.5;     // cytokine degradation rate
  K = 1.0;      // carrying capacity
  WS = 1.0;     // wood smoke exposure level
  H1 = 0.2; // Hill half-saturation constant for P as a function of E_h
  H2 = 0.2; // Hill half-saturation constant for P as a function of E_d
  n1 = 4.0; // Hill exponent for P as a function of E_h
  n2 = 4.0; // Hill exponent for P as a function of E_d
  // Reactions
  R1: -> E_h ; e1 * E_h * (1 - (E_h + E_d) / K);
  R2: -> E_d ; e1 * E_d * (1 - (E_h + E_d) / K);
  R3: E_h -> ; e2 * E_h;
  R4: E_d -> ; e2 * E_d;
  R5: E_h -> E_d; e3 * WS * E_h;
  R6: E_d -> E_h; e4 * E_d;
  R7: -> P; e5 * WS * E_h^n1/(H1^n1+E_h^n1) + e6 * WS * E_d^n2/(H2^n2+E_d^n2);
  R8: P -> ; e7 * P;
end
'''

# Experimental data for pro-inflammatory cytokines
time_exp = np.array([5, 24, 48, 72])
P_exp = np.array([0.0363, 0.4545, 0.5636, 1])

# Create the model
r = te.loada(continuous_exposure_model)

# Simulate the model
result = r.simulate(0, 1000, 10000)

# Extract simulation results
time_sim = result[:, 0]
E_h_sim = result[:, 1]
E_d_sim = result[:, 2]
P_sim = result[:, 3]

# Normalize P_sim to compare with experimental data
P_sim_normalized = P_sim / np.max(P_sim)

# After simulating and normalizing the data, create DataFrames for plotting
simulation_df = pd.DataFrame({
    'Time': time_sim,
    'Healthy Epithelial Cells': E_h_sim,
    'Damaged Epithelial Cells': E_d_sim,
    'Pro-inflammatory Cytokines': P_sim,
    'Normalized Cytokines': P_sim_normalized
})

experimental_df = pd.DataFrame({
    'Time': time_exp,
    'Experimental Data': P_exp
})

# Set up the plot style
sns.set_style("whitegrid")
sns.set_palette("colorblind")

# Create a single plot
fig, ax1 = plt.subplots(figsize=(12, 7))

# Plot simulation results
sns.lineplot(data=simulation_df, x='Time', y='Healthy Epithelial Cells', ax=ax1, label='Healthy Epithelial Cells')
sns.lineplot(data=simulation_df, x='Time', y='Damaged Epithelial Cells', ax=ax1, label='Damaged Epithelial Cells')
sns.lineplot(data=simulation_df, x='Time', y='Pro-inflammatory Cytokines', ax=ax1, label='Pro-inflammatory Cytokines')

# Set axis labels
ax1.set_xlabel('Time (hours)', fontsize=14)
ax1.set_ylabel('Concentration', fontsize=14)

# Set x-axis limits to match the time of interest
ax1.set_xlim(0, 1000)
ax1.legend(fontsize=12)

# Customize tick label sizes
ax1.tick_params(axis='both', which='major', labelsize=12)

# Save the figure in different formats
plt.savefig('epithelial_baseline.png', dpi=300, bbox_inches='tight')
plt.savefig('epithelial_baseline.svg', format='svg', bbox_inches='tight')
plt.savefig('epithelial_baseline.eps', format='eps', dpi=300, bbox_inches='tight')

# Adjust layout and display the plot
plt.tight_layout()

plt.show()

Parameter sensitivity analysis

In [ ]:
#Epithelial-cytokine model
#Parameter scans for parameters e3 and e4
#Produces Figure 5
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Antimony model
continuous_exposure_model = """
model wood_smoke_continuous_exposure
  // Species
  species E_h, E_d, P

   // Parameters
  E_h = 0.25;  // healthy epithelial cells
  E_d = 0.0;  // damaged epithelial cells
  P = 0.0;    // pro-inflammatory cytokines
  e1 = 0.0315;  // natural proliferation rate constant
  e2 = 0.0001;   // natural death rate constant
  e3 = 0.005;   // damage rate constant due to wood smoke
  e4 = 0.002;   // repair rate constant
  e5 = 0.1;     // cytokine production rate constant by healthy cells
  e6 = 0.2;     // cytokine production rate constant by damaged cells
  e7 = 0.5;     // cytokine degradation rate
  K = 1.0;      // carrying capacity
  WS = 1.0;     // wood smoke exposure level
  H1 = 0.2; // Hill half-saturation constant for P as a function of E_h
  H2 = 0.2; // Hill half-saturation constant for P as a function of E_d
  n1 = 4.0; // Hill exponent for P as a function of E_h
  n2 = 4.0; // Hill exponent for P as a function of E_d
  // Reactions
  R1: -> E_h ; e1 * E_h * (1 - (E_h + E_d) / K);
  R2: -> E_d ; e1 * E_d * (1 - (E_h + E_d) / K);
  R3: E_h -> ; e2 * E_h;
  R4: E_d -> ; e2 * E_d;
  R5: E_h -> E_d; e3 * WS * E_h;
  R6: E_d -> E_h; e4 * E_d;
  R7: -> P; e5 * WS * E_h^n1/(H1^n1+E_h^n1) + e6 * WS * E_d^n2/(H2^n2+E_d^n2);
  R8: P -> ; e7 * P;

end
"""

# Create the Tellurium model
r = te.loada(continuous_exposure_model)

# Use a colorblind-friendly palette
colors = sns.color_palette("colorblind")

# Define line styles
line_styles = ['-', '--', '-.', ':']

# Parameter scan for varying damage parameter e3
e3_values = [0.001, 0.002, 0.005, 0.01]
plt.figure(figsize=(10, 5))
for i, e3 in enumerate(e3_values):
    r.resetAll()
    r.e3 = e3
    result = r.simulate(0, 72, 1000, ['time', 'E_d'])
    plt.plot(result[:, 0], result[:, 1], label=f'$e_3={e3}$', color=colors[i % len(colors)], linestyle=line_styles[i % len(line_styles)])
plt.xlabel('Time (hours)', fontsize=14)
plt.ylabel('Damaged Epithelial Cells ($E_d$)', fontsize=14)
plt.legend(fontsize=12)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.xlim(0, 72)
plt.ylim(0, max(result[:, 1]))
#plt.title('Effects of Varying Damage Parameter $e_3$', fontsize=16)
plt.savefig('Effects_of_Varying_Damage_Parameter_e3.png', format='png')
plt.savefig('Effects_of_Varying_Damage_Parameter_e3.svg', format='svg')
plt.savefig('Effects_of_Varying_Damage_Parameter_e3.eps', format='eps')
plt.show()

# Parameter scan for varying repair parameter e4
e4_values = [0.001, 0.002, 0.005, 0.01]  # Various values for repair rate e4
plt.figure(figsize=(10, 5))
for i, e4 in enumerate(e4_values):
    r.resetAll()
    r.e4 = e4
    result = r.simulate(0, 72, 1000, ['time', 'E_d'])
    plt.plot(result[:, 0], result[:, 1], label=f'$e_4={e4}$', color=colors[i % len(colors)], linestyle=line_styles[i % len(line_styles)])
plt.xlabel('Time (hours)', fontsize=14)
plt.ylabel('Damaged Epithelial Cells ($E_d$)', fontsize=14)
plt.legend(fontsize=12)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.xlim(0, 72)
plt.ylim(0, 0.25)
#plt.title('Effects of Varying Repair Parameter $e_4$', fontsize=16)
plt.savefig('Effects_of_Varying_Repair_Parameter_e4.png', format='png')
plt.savefig('Effects_of_Varying_Repair_Parameter_e4.svg', format='svg')
plt.savefig('Effects_of_Varying_Repair_Parameter_e4.eps', format='eps')
plt.show()


Simulations

In [ ]:
#Epithelial-cytokine model
#Wood smoke scenario exploration
#Produces Figure 6
import tellurium as te
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the model with dosing events for one-time exposure
one_time_exposure_model = '''
model wood_smoke_one_time_exposure
  // Species
  species E_h, E_d, P

   // Parameters
  E_h = 1.0;  // healthy epithelial cells
  E_d = 0.0;  // damaged epithelial cells
  P = 0.0;    // pro-inflammatory cytokines
  e1 = 0.0315;  // natural proliferation rate constant
  e2 = 0.0001;   // natural death rate constant
  e3 = 0.005;   // damage rate constant due to wood smoke
  e4 = 0.002;   // repair rate constant
  e5 = 0.1;     // cytokine production rate constant by healthy cells
  e6 = 0.2;     // cytokine production rate constant by damaged cells
  e7 = 0.5;     // cytokine degradation rate
  K = 1.0;      // carrying capacity
  WS = 1.0;     // wood smoke exposure level
  H1 = 0.2; // Hill half-saturation constant for P as a function of E_h
  H2 = 0.2; // Hill half-saturation constant for P as a function of E_d
  n1 = 4.0; // Hill exponent for P as a function of E_h
  n2 = 4.0; // Hill exponent for P as a function of E_d
  // Reactions
  R1: -> E_h ; e1 * E_h * (1 - (E_h + E_d) / K);
  R2: -> E_d ; e1 * E_d * (1 - (E_h + E_d) / K);
  R3: E_h -> ; e2 * E_h;
  R4: E_d -> ; e2 * E_d;
  R5: E_h -> E_d; e3 * WS * E_h;
  R6: E_d -> E_h; e4 * E_d;
  R7: -> P; e5 * WS * E_h^n1/(H1^n1+E_h^n1) + e6 * WS * E_d^n2/(H2^n2+E_d^n2);
  R8: P -> ; e7 * P;


  // Wood smoke dosing event
  E1: at (1 < time < 2): WS = 1.0;
  E2: at (time >= 2): WS = 0.0;
end
'''

# Define the model with dosing events for continuous exposure
continuous_exposure_model = '''
model wood_smoke_continuous_exposure
  // Species
  species E_h, E_d, P

  // Parameters
  E_h = 1.0;  // healthy epithelial cells
  E_d = 0.0;  // damaged epithelial cells
  P = 0.0;    // pro-inflammatory cytokines
  e1 = 0.0315;  // natural proliferation rate constant
  e2 = 0.0001;   // natural death rate constant
  e3 = 0.005;   // damage rate constant due to wood smoke
  e4 = 0.002;   // repair rate constant
  e5 = 0.1;     // cytokine production rate constant by healthy cells
  e6 = 0.2;     // cytokine production rate constant by damaged cells
  e7 = 0.5;     // cytokine degradation rate
  K = 1.0;      // carrying capacity
  WS = 1.0;     // wood smoke exposure level
  H1 = 0.2; // Hill half-saturation constant for P as a function of E_h
  H2 = 0.2; // Hill half-saturation constant for P as a function of E_d
  n1 = 4.0; // Hill exponent for P as a function of E_h
  n2 = 4.0; // Hill exponent for P as a function of E_d
  // Reactions
  R1: -> E_h ; e1 * E_h * (1 - (E_h + E_d) / K);
  R2: -> E_d ; e1 * E_d * (1 - (E_h + E_d) / K);
  R3: E_h -> ; e2 * E_h;
  R4: E_d -> ; e2 * E_d;
  R5: E_h -> E_d; e3 * WS * E_h;
  R6: E_d -> E_h; e4 * E_d;
  R7: -> P; e5 * WS * E_h^n1/(H1^n1+E_h^n1) + e6 * WS * E_d^n2/(H2^n2+E_d^n2);
  R8: P -> ; e7 * P;


  // Wood smoke continuous exposure
  E1: at (time > 1): WS = 1.0;
end
'''

# Define the model with dosing events for recurring exposure
recurring_exposure_model = '''
model wood_smoke_recurring_exposure
  // Species
  species E_h, E_d, P

  // Parameters
  E_h = 1.0;  // healthy epithelial cells
  E_d = 0.0;  // damaged epithelial cells
  P = 0.0;    // pro-inflammatory cytokines
  e1 = 0.0315;  // natural proliferation rate constant
  e2 = 0.0001;   // natural death rate constant
  e3 = 0.005;   // damage rate constant due to wood smoke
  e4 = 0.002;   // repair rate constant
  e5 = 0.1;     // cytokine production rate constant by healthy cells
  e6 = 0.2;     // cytokine production rate constant by damaged cells
  e7 = 0.5;     // cytokine degradation rate
  K = 1.0;      // carrying capacity
  WS = 1.0;     // wood smoke exposure level
  H1 = 0.2; // Hill half-saturation constant for P as a function of E_h
  H2 = 0.2; // Hill half-saturation constant for P as a function of E_d
  n1 = 4.0; // Hill exponent for P as a function of E_h
  n2 = 4.0; // Hill exponent for P as a function of E_d
  // Reactions
  R1: -> E_h ; e1 * E_h * (1 - (E_h + E_d) / K);
  R2: -> E_d ; e1 * E_d * (1 - (E_h + E_d) / K);
  R3: E_h -> ; e2 * E_h;
  R4: E_d -> ; e2 * E_d;
  R5: E_h -> E_d; e3 * WS * E_h;
  R6: E_d -> E_h; e4 * E_d;
  R7: -> P; e5 * WS * E_h^n1/(H1^n1+E_h^n1) + e6 * WS * E_d^n2/(H2^n2+E_d^n2);
  R8: P -> ; e7 * P;


  // Wood smoke recurring dosing events for 3 meals a day
  E1: at (time % 24 >= 8 and time % 24 < 9): WS = 1.0;
  E2: at (time % 24 >= 9 and time % 24 < 13): WS = 0.0;
  E3: at (time % 24 >= 13 and time % 24 < 14): WS = 1.0;
  E4: at (time % 24 >= 14 and time % 24 < 18): WS = 0.0;
  E5: at (time % 24 >= 18 and time % 24 < 19): WS = 1.0;
  E6: at (time % 24 >= 19 and time % 24 < 24): WS = 0.0;
end
'''

# Function to simulate and plot a model
def simulate_and_plot(model_code, title, label_fontsize=14, tick_fontsize=12):
    r = te.loada(model_code)
    result = r.simulate(0, 72, 1000)

    # Use a colorblind-friendly palette
    colors = sns.color_palette("colorblind")
    line_styles = ['-', '--', '-.', ':']

    plt.figure(figsize=(10, 6))
    for i, species in enumerate(['$E_h$', '$E_d$', '$P$']):
        plt.plot(result[:,0], result[:,i+1], label=species, color=colors[i], linestyle=line_styles[i])

    plt.xlabel('Time (hours)', fontsize=label_fontsize)
    plt.ylabel('Normalized Concentration', fontsize=label_fontsize)
    plt.legend(fontsize=label_fontsize)
    plt.xticks(fontsize=tick_fontsize)
    plt.yticks(fontsize=tick_fontsize)
    # plt.title(title, fontsize=label_fontsize)
    # plt.ylim(0, 0.25)
    plt.xlim(0, 72)

    plt.savefig(f'{title}.png', format='png')
    plt.savefig(f'{title}.eps', format='eps')
    plt.show()

# Simulate and plot one-time exposure
simulate_and_plot(one_time_exposure_model, 'Wood Smoke One-Time Exposure Simulation')

# Simulate and plot continuous exposure
simulate_and_plot(continuous_exposure_model, 'Wood Smoke Continuous Exposure Simulation')

# Simulate and plot recurring exposure (3 meals a day)
simulate_and_plot(recurring_exposure_model, 'Wood Smoke Recurring Exposure Simulation (3 meals a day)')

Validation

In [ ]:
#Epithelial-cytokine model
#Plot of experimental data from Calcabrini et al. [1] together with model predictions
#Produces Figure 7
#Calcabrini, A., Meschini, S., Marra, M., Falzano, L., Colone, M., De Berardis, B., ... & Fiorentini, C. (2004). Fine environmental particulate engenders alterations in human lung epithelial A549 cells. Environmental research, 95(1), 82-91.
import tellurium as te
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Model
continuous_exposure_model = '''
model wood_smoke_continuous_exposure
  // Species
  species E_h, E_d, P

   // Parameters
  E_h = 0.25;  // healthy epithelial cells
  E_d = 0.0;  // damaged epithelial cells
  P = 0.0;    // pro-inflammatory cytokines
  e1 = 0.0315;  // natural proliferation rate constant
  e2 = 0.001;   // natural death rate constant
  e3 = 0.005;   // damage rate constant due to wood smoke
  e4 = 0.002;   // repair rate constant
  e5 = 0.1;     // cytokine production rate constant by healthy cells
  e6 = 0.2;     // cytokine production rate constant by damaged cells
  e7 = 0.5;     // cytokine degradation rate
  K = 1.0;      // carrying capacity
  WS = 1.0;     // wood smoke exposure level
  H1 = 0.2; // Hill half-saturation constant for P as a function of E_h
  H2 = 0.2; // Hill half-saturation constant for P as a function of E_d
  n1 = 4.0; // Hill exponent for P as a function of E_h
  n2 = 4.0; // Hill exponent for P as a function of E_d
  // Reactions
  R1: -> E_h ; e1 * E_h * (1 - (E_h + E_d) / K);
  R2: -> E_d ; e1 * E_d * (1 - (E_h + E_d) / K);
  R3: E_h -> ; e2 * E_h;
  R4: E_d -> ; e2 * E_d;
  R5: E_h -> E_d; e3 * WS * E_h;
  R6: E_d -> E_h; e4 * E_d;
  R7: -> P; e5 * WS * E_h^n1/(H1^n1+E_h^n1) + e6 * WS * E_d^n2/(H2^n2+E_d^n2);
  R8: P -> ; e7 * P;

end
'''

# Experimental data
time_exp = np.array([5, 24, 48, 72])
TNF_alpha_exp = np.array([0.07, 0.07, 1.0, 0.07])
IL6_exp = np.array([0.0363, 0.4545, 0.5636, 1.0])
P_exp_sum = np.array([0.045, 0.459, 0.685, 1])  # Sum of normalized TNF-alpha and IL-6

# Create the model and simulate
r = te.loada(continuous_exposure_model)
result = r.simulate(0, 73, 1000)

# Extract and normalize simulation results
time_sim = result[:, 0]
P_sim = result[:, 3]
P_sim_normalized = P_sim / np.max(P_sim)

# Create DataFrames for plotting
simulation_df = pd.DataFrame({
    'Time': time_sim,
    'Normalized Cytokines': P_sim_normalized
})

experimental_df = pd.DataFrame({
    'Time': time_exp,
    'TNF-alpha': TNF_alpha_exp,
    'IL-6': IL6_exp,
    'Sum': P_exp_sum
})

# Set up the plot style
sns.set_style("whitegrid")
sns.set_palette("colorblind")

# First figure - Individual cytokine levels
plt.figure(figsize=(10, 7))
bar_width = 0.35
index = np.arange(len(time_exp))
plt.bar(index - bar_width/2, TNF_alpha_exp, bar_width, label='TNF-alpha', alpha=0.8)
plt.bar(index + bar_width/2, IL6_exp, bar_width, label='IL-6', alpha=0.8)

plt.tick_params(axis='both', which='major', labelsize=12)
plt.xlabel('Time (hours)', fontsize=14)
plt.ylabel('Normalized Concentration', fontsize=14)
plt.xticks(index, time_exp)
plt.legend(fontsize=12)
plt.ylim(0, 1.1)

# Value labels on top of each bar
for i, v in enumerate(TNF_alpha_exp):
    plt.text(i - bar_width/2, v, f'{v:.2f}', ha='center', va='bottom', fontsize=12)
for i, v in enumerate(IL6_exp):
    plt.text(i + bar_width/2, v, f'{v:.2f}', ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.savefig('figure6a.eps', format='eps', dpi=300, bbox_inches='tight')
plt.show()

# Second figure - Experimental Sum of cytokines vs. Simulated P
plt.figure(figsize=(10, 7))
sns.scatterplot(data=experimental_df, x='Time', y='Sum', s=100, label='TNF-alpha + IL-6')
sns.lineplot(data=simulation_df, x='Time', y='Normalized Cytokines', label='Simulated P (normalized)')
plt.xlabel('Time (hours)', fontsize=14)
plt.ylabel('Normalized Concentration', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=12)
plt.legend(fontsize=12)
plt.xlim(0, 73)

plt.tight_layout()
plt.savefig('figure6b.eps', format='eps', dpi=300, bbox_inches='tight')
plt.show()

# M1-M2-Cytokine Submodel

This section implements macrophage polarization dynamics described in Equations 10-14 of the main text. The model tracks:

*  M0 (undifferentiated macrophages)
*  M1 (pro-inflammatory macrophages)
*  M2 (anti-inflammatory macrophages)
*  Cytokines P and A (pro- and anti-inflammatory)

Implementation

In [ ]:
#M1-M2-cytokine model
# Model predictions for various wood smoke exposure events
# Produces Figure 9
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Define the model with continuous exposure
continuous_exposure_model = '''
  // Define species
  species M0, M1, M2, P, A;

    // Define parameters
    m1 = 50; // M0 to M1 differentiation rate
    m2 = 0.1; // M0 to M2 differentiation rate
    m3 = 0.07; // Proliferation rate of M1 and M2 with carrying capacity
    m4 = 0.001; // Natural death rate of M0, M1, and M2
    m5 = 0.05; // M1 to M2 transition rate
    m6 = 0.2; // Production rate of pro-inflammatory cytokine P
    m7 = 0.04; // Degradation rate of P
    m8 = 0.1; // Production rate of anti-inflammatory cytokine A by M2
    m9 = 0.04; // Degradation rate of A
    m10 = 0.1; // Production rate of A by M1
    Hm1 = 0.25; // Half-saturation constant for P
    Hm2 = 0.05; // Half-saturation constant for A
    Hm3 = 0.9; // Half-saturation constant for A inhibition function
    Hm4 = 1.2; // Half-saturation constant for A inhibition of P production
    n1 = 2.5; // Hill coefficient for P production
    n2 = 4; // Hill coefficient for A production
    n3 = 2; // Hill coefficient for A inhibition of P production
    n4 = 2; // Hill coefficient for A inhibition

    WS = 1.0; // Wood smoke exposure level
    K = 1.0; // Carrying capacity for M1 and M2

    // Initial conditions
    M0 = 0.07;
    M1 = 0.07;
    M2 = 0.07;
    P = 0.001;
    A = 0.001;

    // Equations
    J1: M0 -> M1; m1 * (P^n1 / (P^n1 + Hm1^n1)) * M0;
    J2: M0 -> M2; m2 * (A^n2 / (A^n2 + Hm2^n2)) * M0;
    J3: M1 -> M0; m3 * M1 * (1 - (M0 + M1 + M2) / K);
    J4: M2 -> M0; m3 * M2 * (1 - (M0 + M1 + M2) / K);
    J5: M0 -> ; m4 * M0;
    J6: M1 -> ; m4 * M1;
    J7: M2 -> ; m4 * M2;
    J8: M1 -> M2; m5 * (A^n2 / (A^n2 + Hm2^n2)) * M1;
    J9: -> P; m6 * WS * (Hm4^n4 / (A^n4 + Hm4^n4)) * M1;
    J10: P -> ; m7 * P;
    J11: -> A; m8 * (Hm3^n3 / (A^n3 + Hm3^n3)) * M2;
    J12: A -> ; m9 * A;
    J13: -> A; m10 * (Hm3^n3 / (A^n3 + Hm3^n3)) * M1;

'''

# Define the model with dosing events for one-time exposure
one_time_exposure_model = '''
  // Define species
  species M0, M1, M2, P, A;

    // Define parameters
    m1 = 50; // M0 to M1 differentiation rate
    m2 = 0.1; // M0 to M2 differentiation rate
    m3 = 0.07; // Proliferation rate of M1 and M2 with carrying capacity
    m4 = 0.001; // Natural death rate of M0, M1, and M2
    m5 = 0.05; // M1 to M2 transition rate
    m6 = 0.2; // Production rate of pro-inflammatory cytokine P
    m7 = 0.04; // Degradation rate of P
    m8 = 0.1; // Production rate of anti-inflammatory cytokine A by M2
    m9 = 0.04; // Degradation rate of A
    m10 = 0.1; // Production rate of A by M1
    Hm1 = 0.25; // Half-saturation constant for P
    Hm2 = 0.05; // Half-saturation constant for A
    Hm3 = 0.9; // Half-saturation constant for A inhibition function
    Hm4 = 1.2; // Half-saturation constant for A inhibition of P production
    n1 = 2.5; // Hill coefficient for P production
    n2 = 4; // Hill coefficient for A production
    n3 = 2; // Hill coefficient for A inhibition of P production
    n4 = 2; // Hill coefficient for A inhibition

    WS = 1.0; // Wood smoke exposure level
    K = 1.0; // Carrying capacity for M1 and M2

    // Initial conditions
    M0 = 0.07;
    M1 = 0.07;
    M2 = 0.07;
    P = 0.001;
    A = 0.001;

    // Equations
    J1: M0 -> M1; m1 * (P^n1 / (P^n1 + Hm1^n1)) * M0;
    J2: M0 -> M2; m2 * (A^n2 / (A^n2 + Hm2^n2)) * M0;
    J3: M1 -> M0; m3 * M1 * (1 - (M0 + M1 + M2) / K);
    J4: M2 -> M0; m3 * M2 * (1 - (M0 + M1 + M2) / K);
    J5: M0 -> ; m4 * M0;
    J6: M1 -> ; m4 * M1;
    J7: M2 -> ; m4 * M2;
    J8: M1 -> M2; m5 * (A^n2 / (A^n2 + Hm2^n2)) * M1;
    J9: -> P; m6 * WS * (Hm4^n4 / (A^n4 + Hm4^n4)) * M1;
    J10: P -> ; m7 * P;
    J11: -> A; m8 * (Hm3^n3 / (A^n3 + Hm3^n3)) * M2;
    J12: A -> ; m9 * A;
    J13: -> A; m10 * (Hm3^n3 / (A^n3 + Hm3^n3)) * M1;

// Wood smoke dosing event
  E1: at (1 < time < 2): WS = 1.0;
  E2: at (time >= 2): WS = 0.0;
'''

# Define the model with dosing events for recurring exposure
recurring_exposure_model = '''
  // Define species
  species M0, M1, M2, P, A;

  // Define species
  species M0, M1, M2, P, A;

    // Define parameters
    m1 = 50; // M0 to M1 differentiation rate
    m2 = 0.1; // M0 to M2 differentiation rate
    m3 = 0.07; // Proliferation rate of M1 and M2 with carrying capacity
    m4 = 0.001; // Natural death rate of M0, M1, and M2
    m5 = 0.05; // M1 to M2 transition rate
    m6 = 0.2; // Production rate of pro-inflammatory cytokine P
    m7 = 0.04; // Degradation rate of P
    m8 = 0.1; // Production rate of anti-inflammatory cytokine A by M2
    m9 = 0.04; // Degradation rate of A
    m10 = 0.1; // Production rate of A by M1
    Hm1 = 0.25; // Half-saturation constant for P
    Hm2 = 0.05; // Half-saturation constant for A
    Hm3 = 0.9; // Half-saturation constant for A inhibition function
    Hm4 = 1.2; // Half-saturation constant for A inhibition of P production
    n1 = 2.5; // Hill coefficient for P production
    n2 = 4; // Hill coefficient for A production
    n3 = 2; // Hill coefficient for A inhibition of P production
    n4 = 2; // Hill coefficient for A inhibition

    WS = 1.0; // Wood smoke exposure level
    K = 1.0; // Carrying capacity for M1 and M2

    // Initial conditions
    M0 = 0.07;
    M1 = 0.07;
    M2 = 0.07;
    P = 0.001;
    A = 0.001;

    // Equations
    J1: M0 -> M1; m1 * (P^n1 / (P^n1 + Hm1^n1)) * M0;
    J2: M0 -> M2; m2 * (A^n2 / (A^n2 + Hm2^n2)) * M0;
    J3: M1 -> M0; m3 * M1 * (1 - (M0 + M1 + M2) / K);
    J4: M2 -> M0; m3 * M2 * (1 - (M0 + M1 + M2) / K);
    J5: M0 -> ; m4 * M0;
    J6: M1 -> ; m4 * M1;
    J7: M2 -> ; m4 * M2;
    J8: M1 -> M2; m5 * (A^n2 / (A^n2 + Hm2^n2)) * M1;
    J9: -> P; m6 * WS * (Hm4^n4 / (A^n4 + Hm4^n4)) * M1;
    J10: P -> ; m7 * P;
    J11: -> A; m8 * (Hm3^n3 / (A^n3 + Hm3^n3)) * M2;
    J12: A -> ; m9 * A;
    J13: -> A; m10 * (Hm3^n3 / (A^n3 + Hm3^n3)) * M1;

  // Wood smoke recurring dosing events for 3 meals a day
  E1: at (time % 24 >= 8 and time % 24 < 9): WS = 1.0;
  E2: at (time % 24 >= 9 and time % 24 < 13): WS = 0.0;
  E3: at (time % 24 >= 13 and time % 24 < 14): WS = 1.0;
  E4: at (time % 24 >= 14 and time % 24 < 18): WS = 0.0;
  E5: at (time % 24 >= 18 and time % 24 < 19): WS = 1.0;
  E6: at (time % 24 >= 19 and time % 24 < 24): WS = 0.0;
'''

# Function to simulate and plot a model
def simulate_and_plot(model_code, title, label_fontsize=14, tick_fontsize=12):
    r = te.loada(model_code)
    result = r.simulate(0, 73, 10000)

    # Use a colorblind-friendly palette
    colors = sns.color_palette("colorblind")
    line_styles = ['-', '--', '-.', ':']

    plt.figure(figsize=(10, 6))

     # LaTeX labels for species
    latex_labels = {
        'M0': r'$M0$',
        'M1': r'$M1$',
        'M2': r'$M2$',
        'P': r'$P$',
        'A': r'$A$'
    }
    for i, species in enumerate(['M0', 'M1', 'M2', 'P', 'A']):
        plt.plot(result[:,0], result[:,i+1], label=latex_labels[species], color=colors[i % len(colors)], linestyle=line_styles[i % len(line_styles)])

    plt.xlabel('Time (hours)', fontsize=label_fontsize)
    plt.ylabel('Concentration', fontsize=label_fontsize)
    plt.legend(fontsize=label_fontsize)
    plt.xticks(fontsize=tick_fontsize)
    plt.yticks(fontsize=tick_fontsize)
    # plt.title(title, fontsize=label_fontsize)
    # plt.ylim(0, 0.25)
    plt.xlim(0, 73)

#Save figure in different file formats
    plt.savefig(f'{title}.eps', format='eps')
    plt.savefig(f'{title}.eps', format='eps')
    plt.savefig(f'{title}.svg', format='svg')
    plt.show()

# Simulate continuous exposure
simulate_and_plot(continuous_exposure_model, 'Wood Smoke Continuous Exposure Simulation')

# Simulate and plot one-time exposure
simulate_and_plot(one_time_exposure_model, 'Wood Smoke One-Time Exposure Simulation')

# Simulate and plot recurring exposure (3 meals a day)
simulate_and_plot(recurring_exposure_model, 'Wood Smoke Recurring Exposure Simulation (3 meals a day)')

Parameter sensitivity analysis

In [ ]:
#M1-M2-cytokine subsystem parameter scan
# Figures 11 - 14
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns

# Define the updated macrophage-cytokine model
macrophage_cytokine_model = '''
    // Define species
    species M0, M1, M2, P, A;

    // Define parameters
    m1 = 50; // M0 to M1 differentiation rate
    m2 = 0.1; // M0 to M2 differentiation rate
    m3 = 0.07; // Proliferation rate of M1 and M2 with carrying capacity
    m4 = 0.001; // Natural death rate of M0, M1, and M2
    m5 = 0.05; // M1 to M2 transition rate
    m6 = 0.2; // Production rate of pro-inflammatory cytokine P
    m7 = 0.04; // Degradation rate of P
    m8 = 0.1; // Production rate of anti-inflammatory cytokine A by M2
    m9 = 0.04; // Degradation rate of A
    m10 = 0.1; // Production rate of A by M1
    Hm1 = 0.25; // Half-saturation constant for P
    Hm2 = 0.05; // Half-saturation constant for A
    Hm3 = 0.9; // Half-saturation constant for A inhibition function
    Hm4 = 1.2; // Half-saturation constant for A inhibition of P production
    n1 = 2.5; // Hill coefficient for P production
    n2 = 4; // Hill coefficient for A production
    n3 = 2; // Hill coefficient for A inhibition of P production
    n4 = 2; // Hill coefficient for A inhibition

    WS = 1.0; // Wood smoke exposure level
    K = 1.0; // Carrying capacity for M1 and M2

    // Initial conditions
    M0 = 0.07;
    M1 = 0.07;
    M2 = 0.07;
    P = 0.001;
    A = 0.001;

    // Equations
    J1: M0 -> M1; m1 * (P^n1 / (P^n1 + Hm1^n1)) * M0;
    J2: M0 -> M2; m2 * (A^n2 / (A^n2 + Hm2^n2)) * M0;
    J3: M1 -> M0; m3 * M1 * (1 - (M0 + M1 + M2) / K);
    J4: M2 -> M0; m3 * M2 * (1 - (M0 + M1 + M2) / K);
    J5: M0 -> ; m4 * M0;
    J6: M1 -> ; m4 * M1;
    J7: M2 -> ; m4 * M2;
    J8: M1 -> M2; m5 * (A^n2 / (A^n2 + Hm2^n2)) * M1;
    J9: -> P; m6 * WS * (Hm4^n4 / (A^n4 + Hm4^n4)) * M1;
    J10: P -> ; m7 * P;
    J11: -> A; m8 * (Hm3^n3 / (A^n3 + Hm3^n3)) * M2;
    J12: A -> ; m9 * A;
    J13: -> A; m10 * (Hm3^n3 / (A^n3 + Hm3^n3)) * M1;
'''

# Create a Tellurium model
r = te.loada(macrophage_cytokine_model)

# Use a colorblind-friendly palette
colors = sns.color_palette("colorblind")

# Define line styles
line_styles = ['-', '--', '-.', ':']

# Parameter scan for varying proliferation rate m3
m3_values = [0.029, 0.058, 0.116, 0.232]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
for i, m3 in enumerate(m3_values):
    r.resetAll()
    r.m3 = m3
    result = r.simulate(0, 72, 1000, ['time', 'M1', 'M2'])
    ax1.plot(result[:, 0], result[:, 1], label=f'$m_3={m3}$', color=colors[i % len(colors)], linestyle=line_styles[i % len(line_styles)])
    ax2.plot(result[:, 0], result[:, 2], label=f'$m_3={m3}$', color=colors[i % len(colors)], linestyle=line_styles[i % len(line_styles)])

ax1.set_xlabel('Time (hours)', fontsize=14)
ax1.set_ylabel('M1 Concentration', fontsize=14)
ax1.legend(fontsize=10)
ax1.tick_params(axis='both', which='major', labelsize=12)
ax1.set_xlim(0, 72)
ax1.set_ylim(0, 0.3)

ax2.set_xlabel('Time (hours)', fontsize=14)
ax2.set_ylabel('M2 Concentration', fontsize=14)
ax2.legend(fontsize=10)
ax2.tick_params(axis='both', which='major', labelsize=12)
ax2.set_xlim(0, 72)
ax2.set_ylim(0, 0.3)

plt.tight_layout()
plt.savefig('Effects_of_Varying_Proliferation_Rate_m3.eps', format='eps')
plt.savefig('Effects_of_Varying_Proliferation_Rate_m3.svg', format='svg')
plt.show()

# Parameter scan for varying cytokine production rate m6
m6_values = [0.1, 0.2, 0.4, 0.8]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
for i, m6 in enumerate(m6_values):
    r.resetAll()
    r.m6 = m6
    result = r.simulate(0, 72, 1000, ['time', 'M1', 'M2'])
    ax1.plot(result[:, 0], result[:, 1], label=f'$m_6={m6}$', color=colors[i % len(colors)], linestyle=line_styles[i % len(line_styles)])
    ax2.plot(result[:, 0], result[:, 2], label=f'$m_6={m6}$', color=colors[i % len(colors)], linestyle=line_styles[i % len(line_styles)])

ax1.set_xlabel('Time (hours)', fontsize=14)
ax1.set_ylabel('M1 Concentration', fontsize=14)
ax1.legend(fontsize=10)
ax1.tick_params(axis='both', which='major', labelsize=12)
ax1.set_xlim(0, 72)
ax1.set_ylim(0, 0.3)

ax2.set_xlabel('Time (hours)', fontsize=14)
ax2.set_ylabel('M2 Concentration', fontsize=14)
ax2.legend(fontsize=10)
ax2.tick_params(axis='both', which='major', labelsize=12)
ax2.set_xlim(0, 72)
ax2.set_ylim(0, 0.3)

plt.tight_layout()
plt.savefig('Effects_of_Varying_Cytokine_Production_Rate_m6.eps', format='eps')
plt.savefig('Effects_of_Varying_Cytokine_Production_Rate_m6.svg', format='svg')
plt.show()

# Parameter scan for varying cytokine production rate m8
m8_values = [0.065, 0.13, 0.26, 0.52]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
for i, m8 in enumerate(m8_values):
    r.resetAll()
    r.m8 = m8
    result = r.simulate(0, 72, 1000, ['time', 'M1', 'M2'])
    ax1.plot(result[:, 0], result[:, 1], label=f'$m_8={m8}$', color=colors[i % len(colors)], linestyle=line_styles[i % len(line_styles)])
    ax2.plot(result[:, 0], result[:, 2], label=f'$m_8={m8}$', color=colors[i % len(colors)], linestyle=line_styles[i % len(line_styles)])

ax1.set_xlabel('Time (hours)', fontsize=14)
ax1.set_ylabel('M1 Concentration', fontsize=14)
ax1.legend(fontsize=10)
ax1.tick_params(axis='both', which='major', labelsize=12)
ax1.set_xlim(0, 72)
ax1.set_ylim(0, 0.3)

ax2.set_xlabel('Time (hours)', fontsize=14)
ax2.set_ylabel('M2 Concentration', fontsize=14)
ax2.legend(fontsize=10)
ax2.tick_params(axis='both', which='major', labelsize=12)
ax2.set_xlim(0, 72)
ax2.set_ylim(0, 0.3)

plt.tight_layout()
plt.savefig('Effects_of_Varying_Cytokine_Production_Rate_m8.eps', format='eps')
plt.savefig('Effects_of_Varying_Cytokine_Production_Rate_m8.svg', format='svg')
plt.show()

# Parameter scan for varying half-saturation constant Hm2
Hm2_values = [0.25, 0.5, 1.0]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
for i, Hm2 in enumerate(Hm2_values):
    r.resetAll()
    r.Hm2 = Hm2
    result = r.simulate(0, 72, 1000, ['time', 'M1', 'M2', 'P'])
    ax1.plot(result[:, 0], result[:, 1], label=f'$H_{{m2}}={Hm2}$', color=colors[i % len(colors)], linestyle=line_styles[i % len(line_styles)])
    ax2.plot(result[:, 0], result[:, 2], label=f'$H_{{m2}}={Hm2}$', color=colors[i % len(colors)], linestyle=line_styles[i % len(line_styles)])
    #ax3.plot(result[:, 0], result[:, 3], label=f'$H_{{m2}}={Hm2}$', color=colors[i % len(colors)], linestyle=line_styles[i % len(line_styles)])

ax1.set_xlabel('Time (hours)', fontsize=14)
ax1.set_ylabel('M1 Concentration', fontsize=14)
ax1.legend(fontsize=10)
ax1.tick_params(axis='both', which='major', labelsize=12)
ax1.set_xlim(0, 72)
ax1.set_ylim(0, 0.3)

ax2.set_xlabel('Time (hours)', fontsize=14)
ax2.set_ylabel('M2 Concentration', fontsize=14)
ax2.legend(fontsize=10)
ax2.tick_params(axis='both', which='major', labelsize=12)
ax2.set_xlim(0, 72)
ax2.set_ylim(0, 0.3)

# ax3.set_xlabel('Time (hours)', fontsize=14)
# ax3.set_ylabel('P Concentration', fontsize=14)
# ax3.legend(fontsize=10)
# ax3.tick_params(axis='both', which='major', labelsize=12)
# ax3.set_xlim(0, 72)
# ax3.set_ylim(0, max(result[:, 3]))

plt.tight_layout()
plt.savefig('Effects_of_Varying_Half_Saturation_Constant_Hm2.eps', format='eps')
plt.savefig('Effects_of_Varying_Half_Saturation_Constant_Hm2.png', format='png')
plt.show()



Validation

In [ ]:
#M1-M2-cytokine subsystem versus experimental data
# Figure 15
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

r = te.loada('''
    // Define species
    species M0, M1, M2, P, A;

    // Define parameters
    m1 = 50; // M0 to M1 differentiation rate
    m2 = 0.1; // M0 to M2 differentiation rate
    m3 = 0.07; // Proliferation rate of M1 and M2 with carrying capacity
    m4 = 0.001; // Natural death rate of M0, M1, and M2
    m5 = 0.05; // M1 to M2 transition rate
    m6 = 0.2; // Production rate of pro-inflammatory cytokine P
    m7 = 0.04; // Degradation rate of P
    m8 = 0.1; // Production rate of anti-inflammatory cytokine A by M2
    m9 = 0.04; // Degradation rate of A
    m10 = 0.1; // Production rate of A by M1
    h1 = 0.25; // Half-saturation constant for P
    h2 = 0.05; // Half-saturation constant for A
    h3 = 0.9; // Half-saturation constant for A inhibition function
    h4 = 1.2; // Half-saturation constant for A inhibition of P production
    n1 = 2.5; // Hill coefficient for P production
    n2 = 4; // Hill coefficient for A production
    n3 = 2; // Hill coefficient for A inhibition of P production
    n4 = 2; // Hill coefficient for A inhibition

    WS = 1.0; // Wood smoke exposure level
    K = 1.0; // Carrying capacity for M1 and M2

    // Initial conditions
    M0 = 0.07;
    M1 = 0.07;
    M2 = 0.07;
    P = 0.001;
    A = 0.001;

    // Equations
    J1: M0 -> M1; m1 * (P^n1 / (P^n1 + h1^n1)) * M0;
    J2: M0 -> M2; m2 * (A^n2 / (A^n2 + h2^n2)) * M0;
    J3: M1 -> M0; m3 * M1 * (1 - (M0 + M1 + M2) / K);
    J4: M2 -> M0; m3 * M2 * (1 - (M0 + M1 + M2) / K);
    J5: M0 -> ; m4 * M0;
    J6: M1 -> ; m4 * M1;
    J7: M2 -> ; m4 * M2;
    J8: M1 -> M2; m5 * (A^n2 / (A^n2 + h2^n2)) * M1;
    J9: -> P; m6 * WS * (h4^n4 / (A^n4 + h4^n4)) * M1;
    J10: P -> ; m7 * P;
    J11: -> A; m8 * (h3^n3 / (A^n3 + h3^n3)) * M2;
    J12: A -> ; m9 * A;
    J13: -> A; m10 * (h3^n3 / (A^n3 + h3^n3)) * M1;
    ''')

# Experimental Data
time_exp = np.array([24, 48, 72])
A_exp = np.array([0.8929, 0.9157, 1])
P_exp = np.array([0.8522, 0.9831, 1])

# Simulate the model
result = r.simulate(0, 73, 1000)

# Extract simulation results
time_sim = result[:, 0]
M0_sim = result[:, 1]
M1_sim = result[:, 2]
M2_sim = result[:, 3]
P_sim = result[:, 4]
A_sim = result[:, 5]

# Normalize P_sim to compare with experimental data
P_sim_normalized = P_sim / np.max(P_sim)

# Normalize A_sim to compare with experimental data
A_sim_normalized = A_sim / np.max(A_sim)

# Use a colorblind-friendly palette
colors = sns.color_palette("colorblind")

# Define line styles
line_styles = ['-', '--', '-.', ':', '-']


# Create first figure for transients
fig1 = plt.figure(figsize=(10, 6))
ax1 = fig1.add_subplot(111)

# Plot simulation results with colorblind palette and different line styles
variables = [M0_sim, M1_sim, M2_sim, P_sim, A_sim]
labels = ['$M0$', '$M1$', '$M2$', '$P$', '$A$']

for i, (var, label, color, style) in enumerate(zip(variables, labels, colors, line_styles)):
    ax1.plot(time_sim, var, label=label, color=color, linestyle=style)

ax1.set_xlabel('Time (hours)', fontsize=14)
ax1.set_ylabel('Normalized Concentration', fontsize=14)
ax1.legend(fontsize=12)
ax1.grid(True)
ax1.set_xlim(0, 72)
ax1.tick_params(axis='both', which='major', labelsize=12)

plt.tight_layout()
fig1.savefig('Transients.eps', format='eps')
fig1.savefig('Transients.png', format='png')
fig1.savefig('Transients.svg', format='svg')

# Create second figure for P fit
fig2 = plt.figure(figsize=(10, 6))
ax2 = fig2.add_subplot(111)

ax2.scatter(time_exp, P_exp, color=colors[0], label='Experimental P')
ax2.plot(time_sim, P_sim_normalized, color=colors[1], label='Simulated P (normalized)')
ax2.set_xlabel('Time (hours)', fontsize=14)
ax2.set_ylabel('Normalized Concentration', fontsize=14)
ax2.legend(fontsize=12)
ax2.grid(True)
ax2.set_xlim(0, 73)
ax2.tick_params(axis='both', which='major', labelsize=12)

plt.tight_layout()
fig2.savefig('P_fit.eps', format='eps')
fig2.savefig('P_fit.png', format='png')
fig2.savefig('P_fit.svg', format='svg')

# Create third figure for A fit
fig3 = plt.figure(figsize=(10, 6))
ax3 = fig3.add_subplot(111)

ax3.scatter(time_exp, A_exp, color=colors[2], label='Experimental A')
ax3.plot(time_sim, A_sim_normalized, color=colors[3], label='Simulated A (normalized)')
ax3.set_xlabel('Time (hours)', fontsize=14)
ax3.set_ylabel('Normalized Concentration', fontsize=14)
ax3.legend(fontsize=12)
ax3.grid(True)
ax3.set_xlim(0, 73)
ax3.tick_params(axis='both', which='major', labelsize=12)

plt.tight_layout()
fig3.savefig('A_fit.eps', format='eps')
fig3.savefig('A_fit.png', format='png')
fig3.savefig('A_fit.svg', format='svg')

plt.show()


# Full Combined Model

This section combines the epithelial and macrophage submodels into a complete system as described in Equations 15-20 of the main text.

*   Healthy epithelial cells (E_h)
*   Damaged epithelial cells (E_d)
*  M0 (undifferentiated macrophages)
*  M1 (pro-inflammatory macrophages)
*  M2 (anti-inflammatory macrophages)
*  Cytokines P and A (pro- and anti-inflammatory)

Implementation and simulations

In [ ]:
#Full model baseline parameters and wood smoke scenarios
# Figure 17
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns

# Define the full model
model = '''
    // Compartments
    compartment C1;

    // Species
    species Eh in C1; Eh = 1.0;     // Healthy epithelial cells
    species Ed in C1; Ed = 0.0;     // Damaged epithelial cells
    species M1 in C1; M1 = 0.001;    // Recruited M1 macrophages
    species M2 in C1; M2 = 0.001;    // Recruited M2 macrophages
    species M0R in C1; M0R = 0.0;  // Resident M0 macrophages
    species M1R in C1; M1R = 0.0;   // Resident M1 macrophages
    species M2R in C1; M2R = 0.001;   // Resident M2 macrophages, low M2 dominant population
    species P in C1; P = 0.0;       // Pro-inflammatory cytokines
    species A in C1; A = 0.000;     // Anti-inflammatory cytokines

    // Parameters
    e1 = 0.0315;  // Epithelial cell proliferation rate
    e2 = 0.0001;   // Epithelial cell death rate
    e3 = 0.005;   // Healthy epithelial cell damage rate
    e4 = 0.002;   // Damaged epithelial cell repair rate
    e5 = 0.1;     // Healthy epithelial pro-inflammatory cytokine production rate
    e6 = 0.2;     // Damaged epithelial pro-inflammatory cytokine production rate
    He1 = 0.2;    // Hill constant for P production by Eh
    He2 = 0.2;    // Hill constant for P production by Ed
    ne1 = 4;      // Hill coefficient for P production by Eh
    ne2 = 4;      // Hill coefficient for P production by Ed

    m1 = 50;      // M0 to M1 differentiation rate
    m2 = 0.1;     // M0 to M2 differentiation rate
    m4 = 0.02;    // Macrophage decay (death/efflux) rate
    m5 = 0.05;    // M1 to M2 transition rate
    m6 = 0.2;     // M1 pro-inflammatory cytokine production rate
    m7 = 0.04;    // Pro-inflammatory cytokine degradation rate
    m8 = 0.1;     // M2 anti-inflammatory cytokine production rate
    m9 = 0.04;    // M1 anti-inflammatory cytokine production rate
    m10 = 0.04;   // Anti-inflammatory cytokine degradation rate
    Hm1 = 0.25;   // Hill constant for P activation
    Hm2 = 0.05;   // Hill constant for A activation
    Hm3 = 0.9;    // Hill constant for A inhibition
    Hm4 = 1.2;    // Hill constant for A self-inhibition
    nm1 = 2.5;    // Hill coefficient for P activation
    nm2 = 4;      // Hill coefficient for A activation
    nm3 = 2;      // Hill coefficient for A inhibition
    nm4 = 2;      // Hill coefficient for A self-inhibition

    # sM0 = 0.1;    // Influx rate of M0 macrophages
    # muM0 = 0.05;  // Efflux/death rate of M0 macrophages
    k_c = 0.0125; // Concentration of background anti-inflammatory cytokine

    K = 1;        // Carrying capacity
    WS = 1.0;       // Wood smoke exposure (1 = on, 0 = off)
    m11 = 0.2;    // Direct WS activation rate for M1
    m3 = 0.07;    // Resident macrophage proliferation rate
    smr = 0.1;    // Circulating macrophage source rate
    mumr = 0.05;  // Circulating macrophage decay rate
    e10 = 0.0;   // Baseline A production by healthy epithelial cells
    e8 = 1.0;     // M2 enhancement of repair
    e9 = 1.0;    // A enhancement of repair


    // Helper functions for activation/influx
    RM1 := m1*(P^nm1/(Hm1^nm1 + P^nm1)) + m11*WS    // M1 activation function
    RM2 := m2*(A^nm2/(Hm2^nm2 + A^nm2)) + k_c       // M2 activation function

    // Differential equations
    Eh' = e1*(Eh + Ed)*(1 - (Eh + Ed)/K) - e2*Eh - e3*WS*Eh + e4*(1 + e9*(M2 + M2R) + e10*A)*Ed
    Ed' = -e2*Ed + e3*WS*Eh - e4*(1 + e8*(M2 + M2R) + e9*A)*Ed

    // Recruited macrophages
    M1' = WS*(smr*RM1)/(mumr + RM1 + RM2) - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M1
    M2' = WS*(smr*RM2)/(mumr + RM1 + RM2) + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M2

    // Resident macrophages
    M0R' = -m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R  - m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m3*WS*(M1R + M2R)*(1 - (M0R + M1R + M2R)/K) - m4*M0R
    M1R' = m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M1R
    M2R' = m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M2R

    // Cytokine equations including both resident and recruited macrophages
    P' = e5*WS*(Eh^ne1/(He1^ne1 + Eh^ne1)) + e6*WS*(Ed^ne2/(He2^ne2 + Ed^ne2)) + m6*WS*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M1 + M1R) - m7*P
    A' = m8*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M2 + M2R) + m9*(Hm4^nm4/(A^nm4 + Hm4^nm4))*(M1 + M1R) - m10*A + k_c

  # // Wood smoke dosing event
  # E1: at (1 < time < 2): WS = 1.0;
  # E2: at (time >= 2): WS = 0.0;

  # // Wood smoke continuous exposure
  # E1: at (time > 1): WS = 1.0;



  // Wood smoke recurring dosing events for 3 meals a day
  E1: at (time % 24 >= 8 and time % 24 < 9): WS = 1.0;
  E2: at (time % 24 >= 9 and time % 24 < 13): WS = 0.0;
  E3: at (time % 24 >= 13 and time % 24 < 14): WS = 1.0;
  E4: at (time % 24 >= 14 and time % 24 < 18): WS = 0.0;
  E5: at (time % 24 >= 18 and time % 24 < 19): WS = 1.0;
  E6: at (time % 24 >= 19 and time % 24 < 24): WS = 0.0;
'''

# Load the model
r = te.loada(model)

# Run simulation
result = r.simulate(0, 200, 1000)

# Set up the plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
sns.set_style("white")
sns.set_palette("colorblind")
line_styles = ['-', '--', '-.', ':']

# Plot cells
cell_species = ['Eh', 'Ed', 'M1', 'M2', 'M0R', 'M1R', 'M2R']
cell_labels = ['Healthy Epithelial', 'Damaged Epithelial', 'Recruited M1', 'Recruited M2',
               'Resident M0', 'Resident M1', 'Resident M2']

for i, (species_name, label) in enumerate(zip(cell_species, cell_labels)):
    ax1.plot(result[:, 0], result[:, i+1], label=label,
             linestyle=line_styles[i % len(line_styles)])

ax1.set_xlabel('Time', fontsize=14)
ax1.set_ylabel('Concentration', fontsize=14)
ax1.set_xlim(0.0,200)
ax1.legend(fontsize=12)
ax1.grid(False)

# Customize tick label sizes
ax1.tick_params(axis='both', which='major', labelsize=12)

# Plot cytokines
cytokine_species = ['P', 'A']
cytokine_labels = ['Pro-inflammatory Cytokines', 'Anti-inflammatory Cytokines']

for i, (species_name, label) in enumerate(zip(cytokine_species, cytokine_labels)):
    ax2.plot(result[:, 0], result[:, i+8], label=label,
             linestyle=line_styles[i % len(line_styles)])

ax2.set_xlabel('Time', fontsize=14)
ax2.set_ylabel('Concentration', fontsize=14)
ax2.set_xlim(0.0,200)
ax2.legend(fontsize=12)
ax2.grid(False)

# Custom tick label sizes
ax2.tick_params(axis='both', which='major', labelsize=12)

# Adjust layout and show the plot
plt.tight_layout()

#Save figure in different file formats
# plt.savefig('full model_one_exp.png', dpi=300, bbox_inches='tight')
# plt.savefig('full model_one_exp.eps', format='eps', bbox_inches='tight')

# plt.savefig('full model_cont_exp.png', dpi=300, bbox_inches='tight')
# plt.savefig('full model_cont_exp.eps', format='eps', bbox_inches='tight')

plt.savefig('full model_rep_exp.png', dpi=300, bbox_inches='tight')
plt.savefig('full model_rep_exp.eps', format='eps', bbox_inches='tight')
plt.show()

Parameter sensitivity analysis

In [ ]:
#Full model parameter scans simulating antioxidant therapy
#Figure 18

import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns

# Model code
model = '''
    // Compartments
    compartment C1;

    // Species
    species Eh in C1; Eh = 1.0;     // Healthy epithelial cells
    species Ed in C1; Ed = 0.0;     // Damaged epithelial cells
    species M1 in C1; M1 = 0.001;    // Recruited M1 macrophages
    species M2 in C1; M2 = 0.001;    // Recruited M2 macrophages
    species M0R in C1; M0R = 0.0;  // Resident M0 macrophages
    species M1R in C1; M1R = 0.0;   // Resident M1 macrophages
    species M2R in C1; M2R = 0.001;   // Resident M2 macrophages, low M2 dominant population
    species P in C1; P = 0.0;       // Pro-inflammatory cytokines
    species A in C1; A = 0.000;     // Anti-inflammatory cytokines

    // Parameters
    e1 = 0.0315;  // Epithelial cell proliferation rate
    e2 = 0.0001;   // Epithelial cell death rate
    e3 = 0.005;   // Healthy epithelial cell damage rate
    e4 = 0.002;   // Damaged epithelial cell repair rate
    e5 = 0.1;     // Healthy epithelial pro-inflammatory cytokine production rate
    e6 = 0.2;     // Damaged epithelial pro-inflammatory cytokine production rate
    He1 = 0.2;    // Hill constant for P production by Eh
    He2 = 0.2;    // Hill constant for P production by Ed
    ne1 = 4;      // Hill coefficient for P production by Eh
    ne2 = 4;      // Hill coefficient for P production by Ed

    m1 = 50;      // M0 to M1 differentiation rate
    m2 = 0.1;     // M0 to M2 differentiation rate
    m4 = 0.02;    // Macrophage decay (death/efflux) rate
    m5 = 0.05;    // M1 to M2 transition rate
    m6 = 0.2;     // M1 pro-inflammatory cytokine production rate
    m7 = 0.04;    // Pro-inflammatory cytokine degradation rate
    m8 = 0.1;     // M2 anti-inflammatory cytokine production rate
    m9 = 0.04;    // M1 anti-inflammatory cytokine production rate
    m10 = 0.04;   // Anti-inflammatory cytokine degradation rate
    Hm1 = 0.25;   // Hill constant for P activation
    Hm2 = 0.05;   // Hill constant for A activation
    Hm3 = 0.9;    // Hill constant for A inhibition
    Hm4 = 1.2;    // Hill constant for A self-inhibition
    nm1 = 2.5;    // Hill coefficient for P activation
    nm2 = 4;      // Hill coefficient for A activation
    nm3 = 2;      // Hill coefficient for A inhibition
    nm4 = 2;      // Hill coefficient for A self-inhibition

    sM0 = 0.1;    // Influx rate of M0 macrophages
    muM0 = 0.05;  // Efflux/death rate of M0 macrophages
    k_c = 0.0125; // Concentration of background anti-inflammatory cytokine

    K = 1;        // Carrying capacity
    WS = 1.0;     // Wood smoke exposure (1 = on, 0 = off)
    m11 = 0.2;    // Direct WS activation rate for M1
    m3 = 0.07;    // Resident macrophage proliferation rate
    smr = 0.1;    // Circulating macrophage source rate
    mumr = 0.05;  // Circulating macrophage decay rate
    e8 = 0.0;     // Baseline A production by healthy epithelial cells
    e9 = 1.0;     // M2 enhancement of repair
    e10 = 1.0;    // A enhancement of repair

    // Helper functions for activation/influx
    RM1 := m1*(P^nm1/(Hm1^nm1 + P^nm1)) + m11*WS    // M1 activation function
    RM2 := m2*(A^nm2/(Hm2^nm2 + A^nm2)) + k_c       // M2 activation function

    // Differential equations
    Eh' = e1*(Eh + Ed)*(1 - (Eh + Ed)/K) - e2*Eh - e3*WS*Eh + e4*(1 + e9*(M2 + M2R) + e10*A)*Ed
    Ed' = -e2*Ed + e3*WS*Eh - e4*(1 + e9*(M2 + M2R) + e10*A)*Ed

    // Recruited macrophages
    M1' = WS*(smr*RM1)/(mumr + RM1 + RM2) - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M1
    M2' = WS*(smr*RM2)/(mumr + RM1 + RM2) + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M2

    // Resident macrophages
    M0R' = -m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m3*WS*(M1R + M2R)*(1 - (M0R + M1R + M2R)/K) - m4*M0R
    M1R' = m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M1R
    M2R' = m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M2R

    // Cytokine equations including both resident and recruited macrophages
    P' = e5*WS*(Eh^ne1/(He1^ne1 + Eh^ne1)) + e6*WS*(Ed^ne2/(He2^ne2 + Ed^ne2)) + m6*WS*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M1 + M1R) - m7*P
    A' = m8*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M2 + M2R) + m9*(Hm4^nm4/(A^nm4 + Hm4^nm4))*(M1 + M1R) - m10*A + k_c

  # // Wood smoke recurring dosing events for 3 meals a day
  # E1: at (time % 24 >= 8 and time % 24 < 9): WS = 1.0;
  # E2: at (time % 24 >= 9 and time % 24 < 13): WS = 0.0;
  # E3: at (time % 24 >= 13 and time % 24 < 14): WS = 1.0;
  # E4: at (time % 24 >= 14 and time % 24 < 18): WS = 0.0;
  # E5: at (time % 24 >= 18 and time % 24 < 19): WS = 1.0;
  # E6: at (time % 24 >= 19 and time % 24 < 24): WS = 0.0;

  // Wood smoke dosing event
  E1: at (1 < time < 2): WS = 1.0;
  E2: at (time >= 2): WS = 0.0;
'''

# Load the model
r = te.loada(model)

# Set up parameter scans with fold changes
fold_changes = [1.0, 0.2, 0.1, 0.05]  # 1x, 1/5x, 1/10x, 1/20x for pro-inflammatory
anti_fold_changes = [1.0, 5.0, 10.0, 20.0]  # 1x, 5x, 10x, 20x for anti-inflammatory

# Set up plot
sns.set_style("whitegrid")
colors = sns.color_palette("colorblind")
linestyles = ['-', '--', '-.', ':']

# Define species and labels
species = ['Eh', 'Ed', 'M1', 'M2', 'M0R', 'M1R', 'M2R', 'P', 'A']
labels = ['Healthy Epithelial', 'Damaged Epithelial', 'Recruited M1', 'Recruited M2',
          'Resident M0', 'Resident M1', 'Resident M2', 'Pro-inflammatory Cytokines',
          'Anti-inflammatory Cytokines']

# Create subplots
fig, axs = plt.subplots(3, 3, figsize=(15, 15))
axs = axs.ravel()

# Perform parameter scan and plot
for j, (species_name, label) in enumerate(zip(species, labels)):
    ax = axs[j]

    for i in range(len(fold_changes)):
        r.reset()

        pro_scale = fold_changes[i]
        anti_scale = anti_fold_changes[i]

        # Scale pro-inflammatory parameters
        r.e5 = 0.1 * pro_scale   # Healthy epithelial pro-inflammatory production
        r.e6 = 0.2 * pro_scale   # Damaged epithelial pro-inflammatory production
        r.m6 = 0.2 * pro_scale   # M1 pro-inflammatory production
        r.m11 = 0.2 * pro_scale  # Direct WS activation of M1
        r.m1 = 50.0 * pro_scale  # M0 to M1 differentiation rate

        # Scale anti-inflammatory parameters
        r.m8 = 0.1 * anti_scale     # M2 anti-inflammatory production
        r.m9 = 0.04 * anti_scale    # M1 anti-inflammatory production
        r.k_c = 0.0125 * anti_scale # Background anti-inflammatory
        r.e9 = 1.0 * anti_scale     # M2 enhancement of repair
        r.e10 = 1.0 * anti_scale    # A enhancement of repair
        r.m2 = 0.1 * anti_scale     # M0 to M2 differentiation rate

        # Run simulation
        result = r.simulate(0, 200, 1000)

        ax.plot(result[:, 0], result[:, j+1],
                color=colors[i],
                linestyle=linestyles[i],
                label=f'{1/pro_scale:.0f}x ↓ pro-,\n{anti_scale:.0f}x ↑ anti-')

    ax.set_title(label)
    ax.set_xlabel('Time')
    ax.set_ylabel('Concentration')
    ax.legend(fontsize='x-small', loc='best')

plt.tight_layout()
plt.savefig('combined_treatment_effects.png', dpi=300, bbox_inches='tight')
plt.show()

# Create individual plots for each species
for j, (species_name, label) in enumerate(zip(species, labels)):
    plt.figure(figsize=(10, 6))

    for i in range(len(fold_changes)):
        r.reset()

        pro_scale = fold_changes[i]
        anti_scale = anti_fold_changes[i]

        # Scale pro-inflammatory parameters
        r.e5 = 0.1 * pro_scale
        r.e6 = 0.2 * pro_scale
        r.m6 = 0.2 * pro_scale
        # r.m11 = 0.2 * pro_scale
        # r.m1 = 50.0 * pro_scale

        # Scale anti-inflammatory parameters
        r.m8 = 0.1 * anti_scale
        r.m9 = 0.04 * anti_scale
        r.k_c = 0.0125 * anti_scale
        r.e9 = 1.0 * anti_scale
        r.e10 = 1.0 * anti_scale
        # r.m2 = 0.1 * anti_scale

        # Run simulation
        result = r.simulate(0, 200, 1000)

        plt.plot(result[:, 0], result[:, j+1],
                 color=colors[i],
                 linestyle=linestyles[i],
                 label=f'{1/pro_scale:.0f}x reduction in pro-inflammatory,\n{anti_scale:.0f}x increase in anti-inflammatory')

    plt.title(label)
    plt.xlabel('Time')
    plt.ylabel('Concentration')
    plt.legend(fontsize='small', loc='best')
    plt.tight_layout()
    plt.savefig(f'treatment_effect_{species_name}.png', dpi=300, bbox_inches='tight')
    plt.show()

Latin Hypercube Sampling of parameter space

In [ ]:
#Perform LHS on parameter range and select representative sets, as well as narrow ranges by excluding trajectories
#figure 22
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import qmc

# Model code
def run_simulation(params):
    model = '''
    // Compartments
    compartment C1;

    // Species
    species Eh in C1; Eh = 1.0;     // Healthy epithelial cells
    species Ed in C1; Ed = 0.0;     // Damaged epithelial cells
    species M1 in C1; M1 = 0.001;    // Recruited M1 macrophages
    species M2 in C1; M2 = 0.001;    // Recruited M2 macrophages
    species M0R in C1; M0R = 0.0;  // Resident M0 macrophages
    species M1R in C1; M1R = 0.0;   // Resident M1 macrophages
    species M2R in C1; M2R = 0.001;   // Resident M2 macrophages
    species P in C1; P = 0.0;       // Pro-inflammatory cytokines
    species A in C1; A = 0.000;     // Anti-inflammatory cytokines

    // Parameters (default values, will be overridden by params dictionary)
    e1 = 0.0315;  // Epithelial cell proliferation rate
    e2 = 0.0001;   // Epithelial cell death rate
    e3 = 0.005;   // Healthy epithelial cell damage rate
    e4 = 0.002;   // Damaged epithelial cell repair rate
    e5 = 0.1;     // Healthy epithelial pro-inflammatory cytokine production rate
    e6 = 0.2;     // Damaged epithelial pro-inflammatory cytokine production rate
    He1 = 0.2;    // Hill constant for P production by Eh
    He2 = 0.2;    // Hill constant for P production by Ed
    ne1 = 4;      // Hill coefficient for P production by Eh
    ne2 = 4;      // Hill coefficient for P production by Ed

    m1 = 50;      // M0 to M1 differentiation rate
    m2 = 0.1;     // M0 to M2 differentiation rate
    m4 = 0.02;    // Macrophage decay (death/efflux) rate
    m5 = 0.05;    // M1 to M2 transition rate
    m6 = 0.2;     // M1 pro-inflammatory cytokine production rate
    m7 = 0.04;    // Pro-inflammatory cytokine degradation rate
    m8 = 0.1;     // M2 anti-inflammatory cytokine production rate
    m9 = 0.04;    // M1 anti-inflammatory cytokine production rate
    m10 = 0.04;   // Anti-inflammatory cytokine degradation rate
    Hm1 = 0.25;   // Hill constant for P activation
    Hm2 = 0.05;   // Hill constant for A activation
    Hm3 = 0.9;    // Hill constant for A inhibition
    Hm4 = 1.2;    // Hill constant for A self-inhibition
    nm1 = 2.5;    // Hill coefficient for P activation
    nm2 = 4;      // Hill coefficient for A activation
    nm3 = 2;      // Hill coefficient for A inhibition
    nm4 = 2;      // Hill coefficient for A self-inhibition

    sM0 = 0.1;    // Influx rate of M0 macrophages
    muM0 = 0.05;  // Efflux/death rate of M0 macrophages
    k_c = 0.0125; // Concentration of background anti-inflammatory cytokine

    K = 1;        // Carrying capacity
    WS = 1.0;     // Wood smoke exposure (1 = on, 0 = off)
    m11 = 0.2;    // Direct WS activation rate for M1
    m3 = 0.07;    // Resident macrophage proliferation rate
    smr = 0.1;    // Circulating macrophage source rate
    mumr = 0.05;  // Circulating macrophage decay rate
    e8 = 0.0;     // Baseline A production by healthy epithelial cells
    e9 = 1.0;     // M2 enhancement of repair
    e10 = 1.0;    // A enhancement of repair

    // Helper functions for activation/influx
    RM1 := m1*(P^nm1/(Hm1^nm1 + P^nm1)) + m11*WS    // M1 activation function
    RM2 := m2*(A^nm2/(Hm2^nm2 + A^nm2)) + k_c       // M2 activation function

    // Differential equations
    Eh' = e1*(Eh + Ed)*(1 - (Eh + Ed)/K) - e2*Eh - e3*WS*Eh + e4*(1 + e9*(M2 + M2R) + e10*A)*Ed
    Ed' = -e2*Ed + e3*WS*Eh - e4*(1 + e8*(M2 + M2R) + e9*A)*Ed

    // Recruited macrophages
    M1' = WS*(smr*RM1)/(mumr + RM1 + RM2) - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M1
    M2' = WS*(smr*RM2)/(mumr + RM1 + RM2) + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M2

    // Resident macrophages
    M0R' = -m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R  - m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m3*WS*(M1R + M2R)*(1 - (M0R + M1R + M2R)/K) - m4*M0R
    M1R' = m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M1R
    M2R' = m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M2R

    // Cytokine equations
    P' = e5*WS*(Eh^ne1/(He1^ne1 + Eh^ne1)) + e6*WS*(Ed^ne2/(He2^ne2 + Ed^ne2)) + m6*WS*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M1 + M1R) - m7*P
    A' = m8*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M2 + M2R) + m9*(Hm4^nm4/(A^nm4 + Hm4^nm4))*(M1 + M1R) - m10*A + k_c

    # // Wood smoke dosing event - uncomment for pulsed exposure
    # E1: at (1 < time < 3): WS = 1.0;
    # E2: at (time >= 3): WS = 0.0;
    '''

    r = te.loada(model)

    # Set parameters
    for param, value in params.items():
        setattr(r, param, value)

    # Run simulation
    result = r.simulate(0, 200, 1000)
    return result

# Define parameter ranges (min, max)
param_ranges = {
    # Epithelial parameters
    'e1': (0.001,1),    # Epithelial cell proliferation rate
    'e2': (0.0001, 0.01),    # Epithelial cell death rate
    'e3': (0.001, 0.1),      # Healthy epithelial cell damage rate
    'e4': (0.001, 0.1),      # Damaged epithelial cell repair rate
    'e5': (0.001, 0.9),          # Healthy epithelial pro-inflammatory cytokine production
    'e6': (0.001, 0.9),          # Damaged epithelial pro-inflammatory cytokine production
    'e8': (1.0, 100),          # M2 enhancement of repair
    'e9': (1, 100),          # A enhancement of repair

    # Hill function parameters
    'He1': (0.5, 6),         # Hill constant for P production by Eh
    'He2': (0.5, 6),         # Hill constant for P production by Ed
    'Hm1': (0.5, 6),       # Hill constant for P activation
    'Hm2': (0.5, 6),       # Hill constant for A activation
    'Hm3': (0.5, 6),         # Hill constant for A inhibition
    'Hm4': (0.5, 6),         # Hill constant for A self-inhibition

    # Macrophage parameters
    'm1': (0.1, 50),            # M0 to M1 differentiation rate
    'm2': (0.1, 10),          # M0 to M2 differentiation rate
    'm3': (0.0001, 0.1),        # Resident macrophage proliferation rate
    'm4': (0.0001, 0.1),        # Macrophage decay rate
    'm5': (0.001, 0.9),        # M1 to M2 transition rate
    'm6': (0.001, 10),          # M1 pro-inflammatory cytokine production
    'm7': (0.001, 0.1),        # Pro-inflammatory cytokine degradation
    'm8': (0.001, 10),          # M2 anti-inflammatory cytokine production
    'm9': (0.001, 1.0),        # M1 anti-inflammatory cytokine production
    'm10': (0.0001, 0.1),       # Anti-inflammatory cytokine degradation
    'm11': (0.001, 1.0),         # Direct WS activation rate for M1

    # Other parameters
    'smr': (0.01, 0.1),         # Circulating macrophage source rate
    'mumr': (0.001, 0.1),      # Circulating macrophage decay rate
    'k_c': (0.0001, 0.1),   # Background anti-inflammatory cytokine
}

def select_representative_trajectories(all_results, variables, n_representatives=3):
    """
    Select representative trajectories that show different qualitative behaviors

    Parameters:
    - all_results: list of simulation results
    - variables: list of variable names
    - n_representatives: number of representative trajectories to select

    Returns:
    - indices of representative trajectories
    """
    # Initialize dictionary to store features for clustering
    features = {}
    n_samples = len(all_results)

    # Extract key features for each trajectory
    for var_idx, var_name in enumerate(variables):
        # Get all trajectories for this variable
        trajectories = np.array([result[:,var_idx+1] for result in all_results])

        # Calculate features for each trajectory
        max_values = np.max(trajectories, axis=1)
        peak_times = np.argmax(trajectories, axis=1)
        final_values = trajectories[:, -1]

        # Store features
        features[f"{var_name}_max"] = max_values
        features[f"{var_name}_peak_time"] = peak_times
        features[f"{var_name}_final"] = final_values

    # Convert features to matrix
    feature_matrix = np.column_stack([features[key] for key in features.keys()])

    # Normalize features to [0, 1] range
    feature_mins = np.min(feature_matrix, axis=0)
    feature_maxs = np.max(feature_matrix, axis=0)
    feature_range = feature_maxs - feature_mins
    # Avoid division by zero
    feature_range[feature_range == 0] = 1.0
    normalized_features = (feature_matrix - feature_mins) / feature_range

    # Select diverse trajectories
    selected_indices = []
    remaining_indices = list(range(n_samples))

    # Median trajectory
    median_idx = remaining_indices[len(remaining_indices) // 2]
    selected_indices.append(median_idx)
    remaining_indices.remove(median_idx)

    # Find other trajectories that are most different from already selected ones
    while len(selected_indices) < n_representatives and remaining_indices:
        max_min_distance = -1
        farthest_idx = -1

        for idx in remaining_indices:
            # Calculate minimum distance to any selected trajectory
            min_distance = float('inf')
            for selected_idx in selected_indices:
                dist = np.linalg.norm(normalized_features[idx] - normalized_features[selected_idx])
                min_distance = min(min_distance, dist)

            # Update if this is the farthest candidate found
            if min_distance > max_min_distance:
                max_min_distance = min_distance
                farthest_idx = idx

        selected_indices.append(farthest_idx)
        remaining_indices.remove(farthest_idx)

    return selected_indices

def run_simulations_with_filtering(param_sets, max_threshold=1.0, min_samples=50):
    """
    Run simulations and filter out parameter sets that produce extreme values

    Parameters:
    - param_sets: list of parameter dictionaries
    - max_threshold: maximum allowed value for any variable
    - min_samples: minimum number of valid samples to collect

    Returns:
    - list of valid simulation results
    """
    print("Running simulations with filtering...")
    valid_results = []
    rejected_count = 0
    total_tried = 0

    # Process parameter sets until we have enough valid samples
    for params in param_sets:
        total_tried += 1

        # Run simulation
        result = run_simulation(params)

        # Check if any variable exceeds threshold
        exceeds_threshold = False
        for col_idx in range(1, result.shape[1]):  # Skip time column
            if np.max(result[:, col_idx]) > max_threshold:
                exceeds_threshold = True
                break

        if exceeds_threshold:
            rejected_count += 1
        else:
            valid_results.append(result)

        # Report progress
        if total_tried % 10 == 0:
            print(f"Processed {total_tried} parameter sets, accepted {len(valid_results)}, rejected {rejected_count}")

        # Check if we have enough valid samples
        if len(valid_results) >= min_samples:
            break

    # Final report
    print(f"Simulation complete. Total tried: {total_tried}, accepted: {len(valid_results)}, rejected: {rejected_count}")

    return valid_results

def plot_with_highlighted_representatives(all_results, variables, n_representatives=3):
    """
    Plot with highlighted representative trajectories using dashed lines
    """
    # Select representative trajectories
    rep_indices = select_representative_trajectories(all_results, variables, n_representatives)

    # Define colors for representative trajectories
    rep_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, orange, green
    background_color = '#DDDDDD'  # Light gray
    percentile_color = (0.7, 0.7, 0.7)  # Medium gray

    for var_idx, var_name in enumerate(variables):
        plt.figure(figsize=(10, 6))

        # Get time and all trajectories
        time = all_results[0][:,0]
        trajectories = np.array([result[:,var_idx+1] for result in all_results])

        # Plot background trajectories in light gray
        for i in range(len(all_results)):
            if i not in rep_indices:
                plt.plot(time, trajectories[i], color=background_color, alpha=0.2, linewidth=0.3)

        # Calculate and plot percentiles
        median = np.median(trajectories, axis=0)
        p25 = np.percentile(trajectories, 25, axis=0)
        p75 = np.percentile(trajectories, 75, axis=0)

        plt.fill_between(time, p25, p75, color=percentile_color, alpha=0.3,
                        label='25th-75th percentile')

        # Extract representative trajectories
        rep_trajectories = [trajectories[idx] for idx in rep_indices]

        # Plot representative trajectories with dashed lines
        for i, idx in enumerate(rep_indices):
            plt.plot(time, trajectories[idx], color=rep_colors[i],
                    linestyle='--', linewidth=2.0, label='Representative trajectory')

        # Add median curve (solid line)
        plt.plot(time, median, 'k-', linewidth=1.5, label='Median trajectory')

        # Remove duplicate labels in legend
        handles, labels = plt.gca().get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        plt.legend(by_label.values(), by_label.keys(), fontsize=10)

        plt.xlabel('Time (hours)', fontsize=12)
        plt.ylabel(f'{var_name} Concentration', fontsize=12)
        plt.grid(False)

        # Set axis limits
        plt.xlim(0, 200)

        # Calculate max value of representative trajectories
        rep_max = max([np.max(trajectories[idx]) for idx in rep_indices])
        # Calculate 95th percentile of all trajectories
        p95_all = np.percentile(trajectories, 95)
        # Set y-axis limit to ensure all representative trajectories are visible
        # with a small buffer (10%) and a minimum value of 0.1
        plt.ylim(0, max(rep_max * 1.1, p95_all, 0.1))

        # Save plots
        plt.savefig(f'LHS_{var_name}_representatives.png', dpi=300, bbox_inches='tight')
        plt.savefig(f'LHS_{var_name}_representatives.eps', dpi=300, bbox_inches='tight')
        plt.close()

# Main execution code
if __name__ == "__main__":
    # Generate parameter sets
    n_samples_to_generate = 200  # Generate 2x what we want to allow for rejections
    sampler = qmc.LatinHypercube(d=len(param_ranges))
    samples = sampler.random(n=n_samples_to_generate)

    # Scale samples to parameter ranges
    param_names = list(param_ranges.keys())
    all_param_sets = []
    for i in range(n_samples_to_generate):
        param_set = {}
        for j, param in enumerate(param_names):
            min_val, max_val = param_ranges[param]
            param_set[param] = min_val + samples[i,j] * (max_val - min_val)
        all_param_sets.append(param_set)

    # Run simulations with filtering
    max_threshold = 1.0
    min_samples = 50  # Minimum number of valid samples we want
    valid_results = run_simulations_with_filtering(all_param_sets, max_threshold, min_samples)

    # Plot variables of interest, excluding resident macrophages
    variables = ['Eh', 'Ed', 'M1', 'M2', 'P', 'A']  # Excluding M0R, M1R, M2R
    plot_with_highlighted_representatives(valid_results, variables, n_representatives=3)